# DSA 504 — Class 11
## Programmatic Data Access II: APIs and Credentials

**Date:** Wednesday, Oct 7
**Reading:** `requests` library documentation
**Today:** HW3 is assigned at the end of class.

---

> **Instructor note:** the API-calling cells in this notebook require real internet access to run — they will not work in a network-restricted environment. Please run this notebook once yourself before class to confirm the two APIs used (JSONPlaceholder and Open-Meteo) are reachable from the classroom network, since both are free, stable, and require no sign-up or API key for the parts used here.

Class 10 covered reading data from files already sitting on disk. Today covers the other major way real data arrives: requesting it live from a web service — an **API**.

### Learning goals
By the end of this class, you will be able to:
- Explain what an API is and what happens during a request/response cycle
- Make a GET request with the `requests` library and parse a JSON response
- Check a response's status code and handle errors gracefully
- Pass query parameters to filter what an API returns
- Convert an API's JSON response directly into a pandas DataFrame
- Store and use an API key safely, without ever hard-coding it into a script


In [1]:
import requests
import pandas as pd
import os

print(requests.__version__)


2.32.5


## 1. What Is an API?

**API** stands for Application Programming Interface — in the context of web data, it usually means: a web address you can send a request to, which responds with data (often in JSON format) instead of a webpage designed for humans to read.

The basic cycle:
1. Your program sends a **request** to a URL.
2. The server processes it and sends back a **response** — data, plus a **status code** telling you whether it worked.
3. Your program reads the response and does something with it (often: turn it into a DataFrame).

This is exactly the same request/response idea as typing a URL into a browser — the difference is the response is structured data (JSON) meant for a program to read, not a formatted page meant for a human to read.


## 2. Your First GET Request

We'll start with **JSONPlaceholder**, a free fake API built specifically for learning and testing — no sign-up, no API key, and it's designed to always be available.


In [25]:
response = requests.get("https://jsonplaceholder.typicode.com/posts/1")

print("Status code:", response.status_code)
print("Response text:", response.text)


Status code: 200
Response text: {
  "userId": 1,
  "id": 1,
  "title": "sunt aut facere repellat provident occaecati excepturi optio reprehenderit",
  "body": "quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut quas totam\nnostrum rerum est autem sunt rem eveniet architecto"
}


**Reading the status code:** `200` means success. Other common codes you'll run into: `404` (not found — you asked for something that doesn't exist), `401`/`403` (you're not authorized — often a missing or bad API key), `500` (something broke on the server's end, not yours).


In [3]:
# .json() parses the response text directly into a Python dictionary
data = response.json()
print(type(data))
print(data)


<class 'dict'>
{'userId': 1, 'id': 1, 'title': 'sunt aut facere repellat provident occaecati excepturi optio reprehenderit', 'body': 'quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut quas totam\nnostrum rerum est autem sunt rem eveniet architecto'}


**Connecting this back to Class 10:** that dictionary is exactly the same shape as the JSON we worked with last class — and ultimately, the same list-of-dictionaries pattern from Class 3. An API response isn't a new concept; it's JSON arriving over a network instead of from a file.


## 3. Checking for Errors

Real code should never assume a request succeeded. Two common patterns:


In [4]:
# Pattern 1: check status_code manually
response = requests.get("https://jsonplaceholder.typicode.com/posts/1")

if response.status_code == 200:
    print("Success!")
    data = response.json()
else:
    print(f"Request failed with status code {response.status_code}")


Success!


In [5]:
# Pattern 2: ask requests to raise an exception automatically on failure
response = requests.get("https://jsonplaceholder.typicode.com/posts/1")

try:
    response.raise_for_status()   # does nothing if successful, raises an error if not
    data = response.json()
    print("Success!")
except requests.exceptions.HTTPError as e:
    print(f"Request failed: {e}")


Success!


In [6]:
# Seeing this in action: deliberately requesting something that doesn't exist
response = requests.get("https://jsonplaceholder.typicode.com/posts/999999")
print("Status code:", response.status_code)   # 404 -- JSONPlaceholder still "succeeds" at the HTTP level
                                                 # but returns an empty result, which is worth checking for too
print("Response body:", response.text)


Status code: 404
Response body: {}


## 4. Query Parameters — Asking for Specific Data

Most real APIs let you filter what you get back using **query parameters**, instead of downloading everything and filtering in pandas afterward.


In [7]:
# Getting only the posts written by user 1
response = requests.get(
    "https://jsonplaceholder.typicode.com/posts",
    params={"userId": 1}
)

posts = response.json()
print(f"Number of posts returned: {len(posts)}")
print(posts[0])


Number of posts returned: 10
{'userId': 1, 'id': 1, 'title': 'sunt aut facere repellat provident occaecati excepturi optio reprehenderit', 'body': 'quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut quas totam\nnostrum rerum est autem sunt rem eveniet architecto'}


**Why filter at the source instead of downloading everything?** If an API has millions of records, downloading all of them just to throw most away in pandas is slow and wasteful — and some APIs won't even let you; they require you to filter server-side via parameters. Using `params=` is almost always better than requesting everything and filtering afterward.


## 5. Turning an API Response into a DataFrame

Since a typical JSON API response is a list of dictionaries, `pd.DataFrame()` can usually convert it directly, with no manual reshaping.


In [8]:
response = requests.get("https://jsonplaceholder.typicode.com/posts", params={"userId": 1})
posts = response.json()

posts_df = pd.DataFrame(posts)
posts_df.head()


,userId,id,title,body
0,1,1,sunt aut facere repellat provident occaecati e...,quia et suscipit\nsuscipit recusandae consequu...
1,1,2,qui est esse,est rerum tempore vitae\nsequi sint nihil repr...
2,1,3,ea molestias quasi exercitationem repellat qui...,et iusto sed quo iure\nvoluptatem occaecati om...
3,1,4,eum et est occaecati,ullam et saepe reiciendis voluptatem adipisci\...
4,1,5,nesciunt quas odio,repudiandae veniam quaerat sunt sed\nalias aut...


## 6. A Real-World Example: Weather Data

**Open-Meteo** is a free, no-API-key weather API — a good example of a real data source you might combine with our retail sales dataset (e.g., "does weather affect daily sales?").


In [29]:
response = requests.get(
    "https://api.open-meteo.com/v1/forecast",
    params={
        "latitude": 43.1,     # Utica, NY (approximate)
        "longitude": -75.23,
        "current": "temperature_2m,wind_speed_10m"
    }
)
response.raise_for_status()
weather = response.json()
print(weather)


{'latitude': 43.110695, 'longitude': -75.22742, 'generationtime_ms': 2.0798444747924805, 'utc_offset_seconds': 0, 'timezone': 'GMT', 'timezone_abbreviation': 'GMT', 'elevation': 142.0, 'current_units': {'time': 'iso8601', 'interval': 'seconds', 'temperature_2m': '°C', 'wind_speed_10m': 'km/h'}, 'current': {'time': '2026-10-14T12:00', 'interval': 900, 'temperature_2m': 15.9, 'wind_speed_10m': 15.9}}


In [10]:
# Pulling out just the current conditions into a small DataFrame
current = weather["current"]
weather_df = pd.DataFrame([current])
weather_df


,time,interval,temperature_2m,wind_speed_10m
0,2026-10-14T11:45,900,15.9,16.2


**Note the nested structure:** `weather["current"]` is itself a dictionary, nested inside the overall response. Real API responses are very often nested like this (dictionaries inside dictionaries, or lists inside dictionaries) — reading the API's documentation to understand the shape of its response is a normal, necessary step, not something you're expected to guess.


## 7. Handling API Keys and Credentials Safely

Many real APIs (unlike the two used above) require an **API key** — a secret string identifying you, used for billing, rate limits, or access control. Handling these safely is a real, practical skill, not a formality.

**Never do this:**
```python
api_key = "sk-abc123secretkey456"   # NEVER hard-code a real key directly in your script
```

**Why this matters:** if this script is ever committed to Git (which, in this course, is *every* script), the key becomes permanently visible in your repository's history — even if you delete it in a later commit, it's still recoverable. Public GitHub repos get scanned constantly for exposed keys within minutes of being pushed.


In [11]:
# The safe pattern: read the key from an environment variable, never from the script itself
api_key = os.environ.get("WEATHER_API_KEY")

if api_key is None:
    print("No API key found in environment variables.")
else:
    print("API key loaded successfully (not printing the actual value).")


No API key found in environment variables.


### Setting an environment variable

This is done **outside** your Python script, so the key never appears in any file you commit:

**In your terminal (temporary, for the current session only):**
```bash
export WEATHER_API_KEY="your-actual-key-here"      # macOS/Linux
set WEATHER_API_KEY=your-actual-key-here            # Windows
```

**Using a `.env` file (persists across sessions) — the more common real-world approach:**
1. Create a file named `.env` in your project folder containing:
```
WEATHER_API_KEY=your-actual-key-here
```
2. Add `.env` to your `.gitignore` file — **this step is what actually prevents it from ever being committed.**
3. Install `python-dotenv` (`pip install python-dotenv`) and load it at the top of your script:
```python
from dotenv import load_dotenv
load_dotenv()   # reads the .env file and loads it into os.environ

api_key = os.environ.get("WEATHER_API_KEY")
```


In [12]:
%pwd

'/Users/inekwet/Documents/Teaching/DSA 504/Class notes'

In [13]:
# Using a key safely in a request (illustrative pattern -- not an executable call,
# since this requires a real key for a real key-requiring API)
api_key = os.environ.get("SOME_API_KEY")

if api_key:
    headers = {"Authorization": f"Bearer {api_key}"}
    # response = requests.get("https://example-api.com/data", headers=headers)
    print("Ready to make an authenticated request.")
else:
    print("Set SOME_API_KEY as an environment variable before running this.")


Set SOME_API_KEY as an environment variable before running this.


---
## Guided Practice

Work through these using the two APIs introduced above. Both require internet access.


### Exercise 1 — Basic GET request
Send a GET request to `https://jsonplaceholder.typicode.com/users/3`. Print the status code and the user's name and email from the parsed JSON.


In [14]:
# Exercise 1 — your code here



### Exercise 2 — Query parameters
Request all posts by `userId=2` from JSONPlaceholder. Print how many posts were returned, and print just the titles of the first 3.


In [15]:
# Exercise 2 — your code here



### Exercise 3 — Error handling
Deliberately request a nonexistent user, `https://jsonplaceholder.typicode.com/users/99999`. Use `.raise_for_status()` inside a try/except block, and print a clear message if it fails (note: JSONPlaceholder returns 404 with an empty body for this, so you should be able to observe this directly).


In [16]:
# Exercise 3 — your code here



### Exercise 4 — API response to DataFrame
Request all posts (no filter) from JSONPlaceholder, convert the result to a DataFrame, and print how many rows and columns it has.


In [17]:
# Exercise 4 — your code here



### Exercise 5 (stretch) — Real-world weather lookup
Using Open-Meteo, look up the current temperature for a city of your choice (look up its latitude/longitude first). Print the city name and current temperature in a single formatted sentence.


In [18]:
# Exercise 5 — your code here



---
## Solutions

Try each exercise yourself first. These require internet access to run.


---
## Wrap-up

**Recap:** the request/response cycle; making GET requests with `requests.get()`; checking status codes and handling errors with `.raise_for_status()`; filtering server-side with query parameters (`params=`); converting a JSON response directly into a DataFrame; and the safe pattern for handling API keys — environment variables and `.env` files, never hard-coded values.

**Common mistakes to watch for:**
- Assuming a request succeeded without checking the status code
- Hard-coding an API key directly into a script, especially one that gets committed to Git
- Forgetting that `.gitignore` must list `.env` *before* the first commit that would have included it — adding it after the fact doesn't remove it from earlier commit history
- Downloading an entire dataset from an API and filtering it in pandas, when the API already supports filtering via query parameters

**HW3 is assigned today** — it will involve working with an API and/or the file formats from Class 10.

**Before next class (Oct 14):** no class Oct 12 (mid-semester break). Class 12 begins SQL — a different way of asking questions of data, used when it lives in a relational database rather than a file or an API.
